# Methods documentation

In [1]:
from IPython.display import Markdown, display

from tb_macro.document import get_func_notes, build_fixed_params_table, build_prior_ranges_table
from tb_macro.acf import add_acf
from tb_macro.health_system import (
    add_detection,
    add_treatment_flows,
    compute_outcome_props,
    get_outcome_rates,
)
from tb_macro.demography import (
    add_ageing_flows,
    add_entry_births,
    add_replacement_deaths,
    inflate_oldest_death_rates,
    prepare_pop_data_for_entries,
)
from tb_macro.epi import (
    add_infection_flows,
    add_latency_flows,
    add_natural_history,
    add_seeding,
    get_base_model,
    get_latency_age_adj,
    infect_process,
    initialise_pops,
)
from tb_macro.calibration import (
    make_log_likelihood,
    get_latent_log_likelihood,
    get_notification_log_likelihood,
    get_death_log_likelihood,
    get_adult_pulm_prev,
    get_pulm_prev_log_likelihood,
    get_prev_decline_log_likelihood,
    get_infprop_log_likelihood,
)
from tb_macro.parameters import BASE_PARAMS, PARAM_BOUNDS

In [2]:
display(Markdown("## Demographic model constructiuon"))
display(Markdown("### Ageing"))
display(Markdown(get_func_notes(add_ageing_flows)))
display(Markdown("### Oldest age group mortality"))
display(Markdown(get_func_notes(inflate_oldest_death_rates)))
display(Markdown("### Background mortality"))
display(Markdown(get_func_notes(add_replacement_deaths)))
display(Markdown("### Construction of entry rates"))
display(Markdown(get_func_notes(prepare_pop_data_for_entries)))
display(Markdown("### Population growth"))
display(Markdown(get_func_notes(add_entry_births)))

display(Markdown("## Epidemiological model construction"))
display(Markdown("### Compartments and stratifications"))
display(Markdown(get_func_notes(get_base_model)))
display(Markdown("### Initial population"))
display(Markdown(get_func_notes(initialise_pops)))
display(Markdown("### Seeding"))
display(Markdown(get_func_notes(add_seeding)))
display(Markdown("### Force of infection"))
display(Markdown(get_func_notes(infect_process)))
display(Markdown("### Infection"))
display(Markdown(get_func_notes(add_infection_flows)))
display(Markdown("### Latency age bands"))
display(Markdown(get_func_notes(get_latency_age_adj)))
display(Markdown("### Latency"))
display(Markdown(get_func_notes(add_latency_flows)))
display(Markdown("### Natural history"))
display(Markdown(get_func_notes(add_natural_history)))
display(Markdown("### Detection"))
display(Markdown(get_func_notes(add_detection)))
display(Markdown("### Treatment outcome proportions"))
display(Markdown(get_func_notes(compute_outcome_props)))
display(Markdown("### Treatment outcome rates"))
display(Markdown(get_func_notes(get_outcome_rates)))
display(Markdown("### Treatment"))
display(Markdown(get_func_notes(add_treatment_flows)))
display(Markdown("### Active case finding"))
display(Markdown(get_func_notes(add_acf)))

display(Markdown("## Calibration"))
display(Markdown(get_func_notes(make_log_likelihood)))
display(Markdown("### Prevalence of infection"))
display(Markdown(get_func_notes(get_latent_log_likelihood)))
display(Markdown("### Notifications"))
display(Markdown(get_func_notes(get_notification_log_likelihood)))
display(Markdown("### Deaths"))
display(Markdown(get_func_notes(get_death_log_likelihood)))
display(Markdown("### Definition of prevalent disease"))
display(Markdown(get_func_notes(get_adult_pulm_prev)))
display(Markdown("### Prevalence of pulmonary disease"))
display(Markdown(get_func_notes(get_pulm_prev_log_likelihood)))
display(Markdown("### Decline in prevalence"))
display(Markdown(get_func_notes(get_prev_decline_log_likelihood)))
display(Markdown("### Infectiousness of prevalent disease"))
display(Markdown(get_func_notes(get_infprop_log_likelihood)))

## Demographic model constructiuon

### Ageing

The population is stratified into the age groups with lower bounds being
0, 3, 5, 10, 15, 18, 40, 65 years. People move from each group to the next at a
constant rate equal to the reciprocal of the group width, such that
the mean time spent in an age group matches its width in years.

The oldest group has no ageing outflow. Exit from this group
occurs only through death.

### Oldest age group mortality

The death rate in the oldest age group is multiplied by
2. In reality the hazard of death
rises with age, so the population in this open-ended group is
concentrated at its younger end, with only a thin tail at the
oldest ages. The rate taken from the data is the average
hazard weighted by that distribution.

The model applies a single constant hazard to the whole group,
which implies exponential attrition and a heavier old-age tail.
Without inflation, too many people remain in this group relative
to the reported age distribution.

...

Modelled adult prevalence of bacteriologically confirmed pulmonary TB was
compared against the second Vietnamese national prevalence survey
(PLOS One). The target is published per 100,000 population and converted
to a proportion of the adult population.

Being a proportion, this quantity was compared on the log-odds scale with
a standard deviation of 0.2.

### Decline in prevalence

The two survey rounds reported by Nguyen et al. (Emerging Infectious
Diseases) are used to target the decline in prevalence rather than absolute values.
Only the ratio of the later to the earlier estimate is taken, 
so that the target constrains the trend in prevalence while remaining
insensitive to any discrepancy between the quantity these surveys
assess and our definition of prevalence.

The modelled ratio applies the same adult prevalence definition at both
time points. The ratio is strictly positive, so it is compared on the
log scale with a standard deviation of 0.1.

### Infectiousness of prevalent disease

The proportion of prevalent adult TB that is highly infectious was
compared against the equivalent proportion from the second Vietnamese
national prevalence survey.

The numerator is the high infectiousness stratum of the active
compartment and the denominator is total adult prevalence, as defined
above. This target therefore constrains how prevalent disease is
distributed across the infectiousness strata, without further
constraining the overall size of the prevalent pool.

As a proportion, this quantity is compared on the log-odds scale with
a standard deviation of 0.2.

## Fixed parameters

In [3]:
fixed_params = {k: v for k, v in BASE_PARAMS.items() if k not in PARAM_BOUNDS}
build_fixed_params_table(fixed_params)

,value
Parameter,
relative susceptibility of the never-infected,1
relative infectiousness of subclinical TB,0.5
relative infectiousness of low-infectious TB,0.4
progression rate under 5 years,2.4
progression rate ages 5 to 15,2
progression rate ages 15 and over,0.1
proportion of progressions that are high-infectious,0.5
containment rate under 5 years,4.4
containment rate ages 5 to 15,4.4


## Prior ranges

Pass the dict of prior bounds to show.

In [4]:
build_prior_ranges_table(PARAM_BOUNDS)

,lower,upper
Parameter,,
raw transmission rate,12,20
background mixing,0.002,0.04
assortative mixing spread,5,15
parent-child mixing strength,1,2.5
relative susceptibility of contained infection,0.2,0.6
relative susceptibility of cleared or recovered infection,0.5,1
relative susceptibility of children,0.5,1
breakdown rate,0.01,1
clearance rate,0.01,0.1
